In [15]:
import ssl

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [16]:
# macOS Python doesn't use the system keychain for SSL by default,
# so we patch the default context to allow downloading torchvision datasets.
ssl._create_default_https_context = ssl._create_unverified_context

transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),  # MNIST mean and std
    ]
)

train_dataset = datasets.MNIST(
    root="./data", train=True, transform=transform, download=True
)
test_dataset = datasets.MNIST(
    root="./data", train=False, transform=transform, download=True
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples:  {len(test_dataset)}")
print(f"Input shape:   {train_dataset[0][0].shape}")
print(f"Classes:       {train_dataset.classes}")

Train samples: 60000
Test samples:  10000
Input shape:   torch.Size([1, 28, 28])
Classes:       ['0 - zero', '1 - one', '2 - two', '3 - three', '4 - four', '5 - five', '6 - six', '7 - seven', '8 - eight', '9 - nine']


In [ ]:
nn = torch.nn.Sequential(
    torch.nn.Flatten(),
    torch.nn.Linear(28 * 28, 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 64),
    torch.nn.ReLU(),
    torch.nn.Linear(64, 10),
)
nn.to(device)

# the CrossEntropyLoss combines LogSoftmax and NLLLoss in one single class,
# which is more numerically stable than using a plain Softmax followed by a NLLLoss
criterion = torch.nn.CrossEntropyLoss()
# Adam is an optimization algorithm that can be used
# instead of the classical stochastic gradient descent procedure
# to update network weights iteratively based on training data
optimizer = torch.optim.Adam(nn.parameters(), lr=0.001)

num_epochs = 5
for epoch in range(num_epochs):
    # set the model to training mode,
    # which is important for layers like dropout and batch normalization
    nn.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        images = images.view(-1, 28 * 28)  # Flatten the images

        # forward pass
        outputs = nn(images)
        # the loss function expects the raw output (logits) from the model, not the probabilities
        loss = criterion(outputs, labels)

        # before we compute the gradients, we need to zero out the previous gradients
        optimizer.zero_grad()
        # here we compute the gradients of the loss with respect to the model parameters
        loss.backward()
        # this is where the weights are updated based on the computed gradients
        optimizer.step()

    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

# set the model to evaluation mode, which is important for layers like dropout and batch normalization
nn.eval()
# we don't need to compute gradients during evaluation, so we wrap the code in a no_grad() context
with torch.no_grad():
    correct, total = 0, 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        images = images.view(-1, 28 * 28)

        # the model outputs raw scores (logits) for each class,
        # and we take the index of the highest score as the predicted class
        outputs = nn(images)
        # torch.max returns both the maximum value and the index of that value along the specified dimension,
        _, predicted = torch.max(outputs.data, 1)
        # we update the total count of samples and the count of correct predictions
        total += labels.size(0)
        # (predicted == labels) creates a boolean tensor where True indicates a correct prediction,
        # and we sum over all elements to get the total number of correct predictions
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")

Epoch [1/5], Loss: 0.1391
Epoch [2/5], Loss: 0.0449
Epoch [3/5], Loss: 0.1089
Epoch [4/5], Loss: 0.0799
Epoch [5/5], Loss: 0.0125
Test Accuracy: 97.85%
